# Week 1 - Data Cleaning & Feature Engineering
**AI-Powered Data Analysis Internship | Excelerate**

Dataset: SLU Opportunity Wise Data

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load & Explore the Dataset

In [2]:
print("=" * 60)
print("STEP 1: LOADING AND EXPLORING THE DATASET")
print("=" * 60)

df = pd.read_csv('Copy of SLU Opportunity Wise Data - Sheet1.csv')

print(f"\n📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 Column Names:\n{df.columns.tolist()}")
print(f"\n🔢 Data Types:\n{df.dtypes}")
print(f"\n❓ Missing Values:\n{df.isnull().sum()}")
print(f"\n🔁 Duplicate Rows: {df.duplicated().sum()}")

STEP 1: LOADING AND EXPLORING THE DATASET

📊 Dataset Shape: 8558 rows × 16 columns

📋 Column Names:
['Learner SignUp DateTime', 'Opportunity Id', 'Opportunity Name', 'Opportunity Category', 'Opportunity End Date', 'First Name', 'Date of Birth', 'Gender', 'Country', 'Institution Name', 'Current/Intended Major', 'Entry created at', 'Status Description', 'Status Code', 'Apply Date', 'Opportunity Start Date']

🔢 Data Types:
Learner SignUp DateTime    object
Opportunity Id             object
Opportunity Name           object
Opportunity Category       object
Opportunity End Date       object
First Name                 object
Date of Birth              object
Gender                     object
Country                    object
Institution Name           object
Current/Intended Major     object
Entry created at           object
Status Description         object
Status Code                 int64
Apply Date                 object
Opportunity Start Date     object
dtype: object

❓ Missing Values:

## Step 2: Data Cleaning

In [3]:
print("\n" + "=" * 60)
print("STEP 2: DATA CLEANING")
print("=" * 60)

# --- 2a. Fix Corrupted Apply Date ---
print("\n🔧 Fixing corrupted Apply Date values...")
bad_dates_mask = df['Apply Date'].str.contains(r'\d{6,}:', regex=True, na=False)
print(f"   Found {bad_dates_mask.sum()} corrupted Apply Date entries")

# Replace corrupted dates with NaT by setting to NaN
df.loc[bad_dates_mask, 'Apply Date'] = np.nan
print(f"   Corrupted dates set to NaN")

# --- 2b. Parse All Date Columns ---
print("\n📅 Parsing date columns...")
date_cols = ['Learner SignUp DateTime', 'Opportunity End Date', 
             'Entry created at', 'Apply Date', 'Opportunity Start Date', 'Date of Birth']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    null_count = df[col].isnull().sum()
    print(f"   {col}: {null_count} nulls after parsing")

# --- 2c. Standardize Institution Name ---
print("\n🏫 Standardizing Institution Name (case inconsistencies)...")
before_unique = df['Institution Name'].nunique()
df['Institution Name'] = df['Institution Name'].str.strip().str.title()
after_unique = df['Institution Name'].nunique()
print(f"   Unique values reduced from {before_unique} → {after_unique}")

# --- 2d. Standardize Current/Intended Major ---
print("\n📚 Standardizing Major...")
df['Current/Intended Major'] = df['Current/Intended Major'].str.strip().str.title()

# --- 2e. Check Gender and Country ---
print(f"\n⚧  Gender values: {df['Gender'].unique().tolist()}")
print(f"🌍 Top 5 Countries:\n{df['Country'].value_counts().head()}")

# --- 2f. Handle Missing Values ---
print("\n🔧 Handling missing values...")
# Institution Name & Major: only 5 missing — fill with 'Unknown'
df['Institution Name'].fillna('Unknown', inplace=True)
df['Current/Intended Major'].fillna('Unknown', inplace=True)
# Opportunity Start Date: 3794 missing — keep as NaT (intentionally missing for some opportunity types)
missing_osd = df['Opportunity Start Date'].isnull().sum()
print(f"   Opportunity Start Date: {missing_osd} NaT values retained (likely intentional for some categories)")
print(f"   Institution Name & Major NaNs: filled with 'Unknown'")

# --- 2g. Validate Status Codes ---
print("\n✅ Validating Status Codes...")
status_mapping = {
    1010: 'Applied', 1020: 'Waitlisted', 1030: 'Rejected',
    1040: 'Withdraw', 1050: 'Started', 1060: 'Dropped Out',
    1070: 'Team Allocated', 1080: 'Started', 1120: 'Rewards Award'
}
print(f"   Status Descriptions found: {df['Status Description'].unique().tolist()}")
print(f"   Status Code range: {df['Status Code'].min()} – {df['Status Code'].max()}")


STEP 2: DATA CLEANING

🔧 Fixing corrupted Apply Date values...
   Found 107 corrupted Apply Date entries
   Corrupted dates set to NaN

📅 Parsing date columns...
   Learner SignUp DateTime: 295 nulls after parsing
   Opportunity End Date: 1262 nulls after parsing
   Entry created at: 0 nulls after parsing
   Apply Date: 307 nulls after parsing
   Opportunity Start Date: 4637 nulls after parsing
   Date of Birth: 0 nulls after parsing

🏫 Standardizing Institution Name (case inconsistencies)...
   Unique values reduced from 2089 → 1818

📚 Standardizing Major...

⚧  Gender values: ['Female', 'Male', "Don't want to specify", 'Other']
🌍 Top 5 Countries:
Country
United States    3976
India            2836
Nigeria           760
Ghana             275
Pakistan          219
Name: count, dtype: int64

🔧 Handling missing values...
   Opportunity Start Date: 4637 NaT values retained (likely intentional for some categories)
   Institution Name & Major NaNs: filled with 'Unknown'

✅ Validating Statu

## Step 3: Feature Engineering

In [4]:
print("\n" + "=" * 60)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 60)

# --- Feature 1: Age of Learner ---
print("\n🎂 Feature 1: Age of Learner")
reference_date = pd.Timestamp('2024-01-01')
df['Age'] = ((reference_date - df['Date of Birth']).dt.days / 365.25).round(1)
df['Age'] = df['Age'].clip(lower=10, upper=80)  # Remove extreme outliers
print(f"   Age range: {df['Age'].min():.0f} – {df['Age'].max():.0f} years")
print(f"   Mean age: {df['Age'].mean():.1f} years")

# --- Feature 2: Signup Month & Season ---
print("\n📅 Feature 2: Signup Month & Season")
df['Signup_Month'] = df['Learner SignUp DateTime'].dt.month
df['Signup_Year'] = df['Learner SignUp DateTime'].dt.year
df['Signup_DayOfWeek'] = df['Learner SignUp DateTime'].dt.day_name()

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

df['Signup_Season'] = df['Signup_Month'].apply(get_season)
print(f"   Signup year range: {df['Signup_Year'].min()} – {df['Signup_Year'].max()}")
print(f"   Season distribution:\n{df['Signup_Season'].value_counts()}")

# --- Feature 3: Days to Apply (Engagement Speed) ---
print("\n⏱️  Feature 3: Days to Apply (Signup → Application)")
df['Days_to_Apply'] = (df['Apply Date'] - df['Learner SignUp DateTime']).dt.days
df['Days_to_Apply'] = df['Days_to_Apply'].clip(lower=0)  # Remove negative values
print(f"   Mean days to apply: {df['Days_to_Apply'].mean():.1f} days")
print(f"   Median days to apply: {df['Days_to_Apply'].median():.1f} days")

# --- Feature 4: Opportunity Duration ---
print("\n📆 Feature 4: Opportunity Duration (days)")
df['Opportunity_Duration_Days'] = (df['Opportunity End Date'] - df['Opportunity Start Date']).dt.days
print(f"   Mean opportunity duration: {df['Opportunity_Duration_Days'].mean():.1f} days")

# --- Feature 5: Engagement Lag (Apply → Opportunity Start) ---
print("\n🔗 Feature 5: Engagement Lag (Apply Date → Opportunity Start Date)")
df['Engagement_Lag_Days'] = (df['Opportunity Start Date'] - df['Apply Date']).dt.days
df['Engagement_Lag_Days'] = df['Engagement_Lag_Days'].clip(lower=0)
print(f"   Mean engagement lag: {df['Engagement_Lag_Days'].mean():.1f} days")

# --- Feature 6: Encode Gender (Binary) ---
print("\n⚧  Feature 6: Binary Gender Encoding")
df['Gender_Encoded'] = df['Gender'].map({
    'Female': 1, 'Male': 0, "Don't want to specify": -1, 'Other': -1
})
print(f"   Encoded distribution:\n{df['Gender_Encoded'].value_counts()}")

# --- Feature 7: Is_SLU (Saint Louis University student) ---
print("\n🏫 Feature 7: Is_SLU Flag")
df['Is_SLU'] = df['Institution Name'].str.contains('Saint Louis', case=False, na=False).astype(int)
print(f"   SLU students: {df['Is_SLU'].sum()} / {len(df)}")

# --- Feature 8: Application Success Flag ---
print("\n✅ Feature 8: Application Success Flag")
success_statuses = ['Started', 'Team Allocated', 'Rewards Award']
df['Is_Successful'] = df['Status Description'].isin(success_statuses).astype(int)
print(f"   Successful applications: {df['Is_Successful'].sum()} ({df['Is_Successful'].mean()*100:.1f}%)")

# --- Feature 9: Opportunity Category Encoded ---
print("\n📂 Feature 9: Opportunity Category One-Hot Encoding")
cat_dummies = pd.get_dummies(df['Opportunity Category'], prefix='Cat')
df = pd.concat([df, cat_dummies], axis=1)
print(f"   Categories encoded: {df['Opportunity Category'].unique().tolist()}")


STEP 3: FEATURE ENGINEERING

🎂 Feature 1: Age of Learner
   Age range: 10 – 57 years
   Mean age: 24.2 years

📅 Feature 2: Signup Month & Season
   Signup year range: 2023.0 – 2024.0
   Season distribution:
Signup_Season
Winter    3473
Summer    2549
Fall      1552
Spring     984
Name: count, dtype: int64

⏱️  Feature 3: Days to Apply (Signup → Application)
   Mean days to apply: 56.7 days
   Median days to apply: 4.0 days

📆 Feature 4: Opportunity Duration (days)
   Mean opportunity duration: 372.7 days

🔗 Feature 5: Engagement Lag (Apply Date → Opportunity Start Date)
   Mean engagement lag: 33.3 days

⚧  Feature 6: Binary Gender Encoding
   Encoded distribution:
Gender_Encoded
 0    5018
 1    3522
-1      18
Name: count, dtype: int64

🏫 Feature 7: Is_SLU Flag
   SLU students: 4385 / 8558

✅ Feature 8: Application Success Flag
   Successful applications: 4072 (47.6%)

📂 Feature 9: Opportunity Category One-Hot Encoding
   Categories encoded: ['Course', 'Competition', 'Internship', '

## Step 4 & 5: Data Validation Summary & Save Dataset

In [6]:
print("\n" + "=" * 60)
print("STEP 4: DATA VALIDATION SUMMARY")
print("=" * 60)
print(f"\n✅ Final dataset shape: {df.shape}")
print(f"✅ Missing values after cleaning:\n{df[['Institution Name','Current/Intended Major','Apply Date']].isnull().sum()}")
print(f"\n✅ New Features Created:")
new_features = ['Age', 'Signup_Month', 'Signup_Year', 'Signup_Season', 'Signup_DayOfWeek',
                'Days_to_Apply', 'Opportunity_Duration_Days', 'Engagement_Lag_Days',
                'Gender_Encoded', 'Is_SLU', 'Is_Successful']
for f in new_features:
    print(f"   • {f}")
print("   • Cat_Course, Cat_Competition, Cat_Internship, Cat_Event, Cat_Engagement")

print("\n" + "=" * 60)
print("STEP 5: SAVING CLEANED DATASET")
print("=" * 60)

df.to_csv('Cleaned_Preprocessed_Dataset_Week1.csv', index=False)
print("\n✅ Cleaned dataset saved as: Cleaned_Preprocessed_Dataset_Week1.csv")
print(f"   Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n🎉  Data Cleaning & Feature Engineering Complete!")


STEP 4: DATA VALIDATION SUMMARY

✅ Final dataset shape: (8558, 32)
✅ Missing values after cleaning:
Institution Name            0
Current/Intended Major      0
Apply Date                307
dtype: int64

✅ New Features Created:
   • Age
   • Signup_Month
   • Signup_Year
   • Signup_Season
   • Signup_DayOfWeek
   • Days_to_Apply
   • Opportunity_Duration_Days
   • Engagement_Lag_Days
   • Gender_Encoded
   • Is_SLU
   • Is_Successful
   • Cat_Course, Cat_Competition, Cat_Internship, Cat_Event, Cat_Engagement

STEP 5: SAVING CLEANED DATASET

✅ Cleaned dataset saved as: Cleaned_Preprocessed_Dataset_Week1.csv
   Final shape: 8558 rows × 32 columns

🎉  Data Cleaning & Feature Engineering Complete!


## Visualizations
Let's create some visual representations of our dataset and engineered features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (15, 10)

# Create a 2x3 subplot grid
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.tight_layout(pad=6.0)

# 1. Age Distribution
sns.histplot(df['Age'].dropna(), bins=30, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution of Learners')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')

# 2. Gender Distribution
gender_counts = df['Gender'].value_counts()
axes[0, 1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
axes[0, 1].set_title('Gender Distribution')

# 3. Top 5 Countries
top_countries = df['Country'].value_counts().head(5)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[0, 2], palette='viridis')
axes[0, 2].set_title('Top 5 Countries by Leaners')
axes[0, 2].set_xlabel('Number of Learners')
axes[0, 2].set_ylabel('')

# 4. Signup Season
sns.countplot(data=df, x='Signup_Season', order=['Spring', 'Summer', 'Fall', 'Winter'], ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Signups by Season')
axes[1, 0].set_xlabel('Season')
axes[1, 0].set_ylabel('Count')

# 5. Application Success Rate
success_counts = df['Is_Successful'].map({1: 'Successful', 0: 'Unsuccessful'}).value_counts()
axes[1, 1].pie(success_counts, labels=success_counts.index, autopct='%1.1f%%', startangle=90, colors=['#66b3ff','#ff9999'])
axes[1, 1].set_title('Application Success Rate')

# 6. Opportunity Category
cat_cols = ['Cat_Course', 'Cat_Competition', 'Cat_Internship', 'Cat_Event', 'Cat_Engagement']
cat_counts = df[cat_cols].sum().sort_values(ascending=False)
sns.barplot(x=cat_counts.values, y=[c.replace('Cat_', '') for c in cat_counts.index], ax=axes[1, 2], palette='coolwarm')
axes[1, 2].set_title('Opportunity Categories')
axes[1, 2].set_xlabel('Count')
axes[1, 2].set_ylabel('')

plt.show()